# Course 2 · Week 2 — Hands-on: Backpropagation + Softmax for Multi-Class

Last week you ran a forward pass with weights handed to you. This week you'll **train the weights yourself** — by implementing the algorithm that produces them: **backpropagation**.

By the end you'll have:

1. Implemented softmax (the "spread these scores into a probability distribution" function)
2. Implemented categorical cross-entropy loss
3. Implemented forward pass for a 2-layer network (with sigmoid hidden + softmax output)
4. Implemented the backward pass — the actual gradient calculations
5. Trained on 3 Gaussian blobs and watched the loss drop from ~1.10 to ~0.003
6. (Stretch) Swapped sigmoid for ReLU — same code structure, different convergence

Estimated time: **60–75 minutes.** Hardest notebook in Course 2 — backprop is conceptually heavy.


## Setup — three Gaussian blobs

Three classes in 2D. Each blob is a cluster of 50 points. Easy to separate visually, but the model has to learn the boundaries from the data alone.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(2)
m_per_class = 50
n_classes = 3

# Three Gaussian blobs in 2D
centers = np.array([[-2.0, 0.0], [+2.0, 0.0], [0.0, +2.5]])
X_list, y_list = [], []
for k, c in enumerate(centers):
    X_list.append(np.random.randn(m_per_class, 2) * 0.6 + c)
    y_list.append(np.full(m_per_class, k))
X = np.vstack(X_list)
y = np.concatenate(y_list)
m = len(y)
print(f"X shape {X.shape}, y shape {y.shape}, classes: {sorted(set(y))}")

def one_hot(y, K):
    Y = np.zeros((len(y), K))
    Y[np.arange(len(y)), y] = 1.0
    return Y
Y = one_hot(y, n_classes)
print(f"Y first row (y[0]={y[0]}): {Y[0]}")


You should see three colored clusters: red, green, blue. They're far apart enough that any reasonable classifier should hit ~100%. Our job: build that classifier from scratch and watch it learn.


In [ ]:
colors = ["#ef4444", "#10b981", "#3b82f6"]
plt.figure(figsize=(7, 5))
for k in range(n_classes):
    plt.scatter(X[y==k, 0], X[y==k, 1], color=colors[k], s=50, alpha=0.7, label=f"class {k}")
plt.xlabel("$x_1$"); plt.ylabel("$x_2$"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Three Gaussian blobs — predict the class")
plt.show()


## Quick recap

Two new ideas this week:

### Softmax — for multi-class output

For a binary problem (cat vs not-cat), one sigmoid output suffices: probability of "cat". For multi-class (cat / dog / bird), you need K outputs that **sum to 1** — a probability distribution.

That's what softmax does. Take K real-valued scores `z_1, ..., z_K`. Map them to:

$$\text{softmax}(\vec z)_k = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

Bigger `z` → bigger probability. All probabilities positive (because `exp` is positive). All sum to 1 (because we divided by the sum).

### Backpropagation — for training

Forward propagation says "given weights, predict outputs". Backprop says "given outputs, weights, and the true labels, compute the gradient of loss with respect to *every weight*." Once you have those gradients, gradient descent does the rest.

The math is "just the chain rule from calculus, applied carefully," but executed by hand it's a lot to track. Don't be discouraged if it doesn't click on the first read — most people need 2–3 passes.


## Exercise 1 — softmax

Implement softmax. Use the **stable** trick: subtract the row-max from each row before exponentiating.

```python
z_shift = z - z.max(axis=1, keepdims=True)
```

Then `exp` the shifted values and divide each row by its sum.

Why the shift? Because `exp(1000)` is a huge number that overflows. After shifting so the max is 0, the largest exp is `exp(0) = 1`. Mathematically identical (we proved softmax is translation-invariant in the recap). Numerically stable.


In [ ]:
def softmax(z):
    """Convert any vector of real numbers into a probability distribution.

    Math: softmax(z)_k = exp(z_k) / sum_j exp(z_j)

    For a batch input shape (m, K), apply along axis=1 (rows).
    Numerically stable trick: subtract z.max from each row before exp,
    so we never compute exp() of huge positive numbers.
    """
    # TODO: implement using the stable form
    return np.zeros_like(z)


# Sanity check
sm = softmax(np.array([[2.0, 1.0, 0.0]]))
print(f"softmax([2, 1, 0]) = {np.round(sm, 4)}")
print(f"sums to: {sm.sum():.4f}")
assert abs(sm.sum() - 1.0) < 1e-9
assert abs(sm[0, 0] - 0.6652) < 1e-3
print("✓ softmax() works")


## Exercise 2 — cross-entropy loss

For each example with one-hot label `Y`, the per-example loss is:

$$L_i = -\sum_{k=1}^K Y_{ik} \log p_{ik}$$

Since `Y` is one-hot, only one term in the sum is nonzero — it picks out `-log(p_correct)`. Average across all examples.

In numpy: `-(Y * np.log(p)).sum(axis=1).mean()` — element-wise multiply, sum across classes, average across examples.

Anchored: a uniform-prediction model on K=3 classes has loss `log(3) ≈ 1.0986`. That's our baseline; trained loss should drop dramatically below this.


In [ ]:
def loss(p, Y):
    """Categorical cross-entropy.

    p: predicted probabilities, shape (m, K)
    Y: one-hot true labels, shape (m, K)

    For each example, the loss is -log(probability assigned to the true class).
    Average across all examples.
    """
    eps = 1e-15
    p = np.clip(p, eps, 1.0)
    # TODO: implement
    return 0.0


# Sanity: uniform predictions for 3 classes → loss = log(3) ≈ 1.0986
uniform = np.full((m, 3), 1/3)
J_uniform = loss(uniform, Y)
print(f"loss(uniform, Y) = {J_uniform:.4f}   expected ≈ {np.log(3):.4f}")
assert abs(J_uniform - np.log(3)) < 0.01
print("✓ loss() works")


## Exercise 3 — initialize and forward

Two fresh weight matrices, plus zero biases. Sigmoid for the hidden layer, softmax for the output. Same forward pattern as last week, just with softmax at the end.

The function should return `(z1, a1, z2, p)`. We need `a1` for backprop.


In [ ]:
# Initialize a 2 → 8 → 3 network
np.random.seed(7)
n_h = 8
W1 = np.random.randn(2, n_h) * 0.5
b1 = np.zeros(n_h)
W2 = np.random.randn(n_h, n_classes) * 0.5
b2 = np.zeros(n_classes)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def forward(X, W1, b1, W2, b2):
    """Forward pass through 2 → n_h → n_classes network.

    Return (z1, a1, z2, p) — we need a1 for backprop.
    """
    # TODO: compute z1, a1, z2, p (use softmax for p, sigmoid for a1)
    return None, None, None, None


z1, a1, z2, p = forward(X, W1, b1, W2, b2)
print(f"z1 shape {z1.shape if z1 is not None else None}, a1 shape {a1.shape if a1 is not None else None}")
print(f"z2 shape {z2.shape if z2 is not None else None}, p shape {p.shape if p is not None else None}")
print(f"Initial loss: {loss(p, Y):.4f}   expected near log(3) = {np.log(3):.4f}")


## Exercise 4 — backpropagation

This is the hard one. Implement the chain of formulas from the recap. Start with `dz2 = (p - Y) / m` and walk back layer by layer.

The algebra:

```
dz2 = (p - Y) / m              # shape (m, K)
dW2 = a1.T @ dz2               # shape (n_h, K)
db2 = dz2.sum(axis=0)          # shape (K,)
da1 = dz2 @ W2.T               # shape (m, n_h)
dz1 = da1 * a1 * (1 - a1)      # sigmoid derivative
dW1 = X.T @ dz1                # shape (n_in, n_h)
db1 = dz1.sum(axis=0)          # shape (n_h,)
```

The shape-matching is the most reliable bug-catcher: if your output shapes don't match what you expect, something's transposed.


In [ ]:
def backward(X, Y, z1, a1, z2, p, W2):
    """Compute gradients via backpropagation.

    Math (the tidy form for softmax + cross-entropy):
        dz2 = (p - Y) / m            ← shape (m, K)
        dW2 = a1.T @ dz2             ← shape (n_h, K)
        db2 = dz2.sum(axis=0)        ← shape (K,)
        da1 = dz2 @ W2.T             ← shape (m, n_h)
        dz1 = da1 * a1 * (1 - a1)    ← sigmoid derivative
        dW1 = X.T @ dz1              ← shape (n_in, n_h)
        db1 = dz1.sum(axis=0)        ← shape (n_h,)

    Returns dW1, db1, dW2, db2.
    """
    m_local = len(X)
    # TODO: implement the chain of formulas above
    dW1 = np.zeros_like(W1)
    db1 = np.zeros_like(b1)
    dW2 = np.zeros_like(W2)
    db2 = np.zeros_like(b2)
    return dW1, db1, dW2, db2


# Quick sanity-check: gradient shapes match parameter shapes
dW1, db1, dW2, db2 = backward(X, Y, z1, a1, z2, p, W2)
assert dW1.shape == W1.shape and db1.shape == b1.shape
assert dW2.shape == W2.shape and db2.shape == b2.shape
print("✓ backward() returns the right shapes")


## Exercise 5 — train it

Now glue it together: forward, compute loss, backward, update weights. Repeat 2000 times.

After training, accuracy should be 100% (these blobs are well-separated). Loss should drop from ~1.1 (random) to ~0.003.


In [ ]:
# Train the network
np.random.seed(7)
W1 = np.random.randn(2, n_h) * 0.5; b1 = np.zeros(n_h)
W2 = np.random.randn(n_h, n_classes) * 0.5; b2 = np.zeros(n_classes)

alpha = 0.5
n_iters = 2000
history = []
for step in range(n_iters):
    # TODO: forward pass, compute loss, backward pass, update parameters
    pass

# Re-run forward to evaluate
z1, a1, z2, p = forward(X, W1, b1, W2, b2)
acc = float((p.argmax(axis=1) == y).mean()) if p is not None else 0.0
print(f"Final loss: {history[-1] if history else 'n/a'}")
print(f"Final accuracy: {acc:.4f}   expected ≈ 1.0000")
assert acc > 0.95, "Training should reach near-100% accuracy on these 3 well-separated blobs"
print("✓ training works")

if history:
    plt.figure(figsize=(7, 3.5))
    plt.plot(history, color="#4338ca")
    plt.xlabel("iteration"); plt.ylabel("loss"); plt.yscale("log")
    plt.title("Training loss"); plt.grid(alpha=0.3)
    plt.show()


## Visualize the decision regions

Sample a grid, predict, color by the argmax class. You should see three colored regions corresponding to the three blobs. The boundaries between regions are the actual surfaces where the network is uncertain (probabilities tied between two classes).


In [ ]:
# Plot the decision regions
xs1, xs2 = np.meshgrid(np.linspace(-4, 4, 100), np.linspace(-3, 5, 100))
grid = np.column_stack([xs1.ravel(), xs2.ravel()])
_, _, _, grid_p = forward(grid, W1, b1, W2, b2)
grid_pred = grid_p.argmax(axis=1).reshape(xs1.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xs1, xs2, grid_pred, levels=[-0.5, 0.5, 1.5, 2.5], colors=["#fef2f2", "#dcfce7", "#dbeafe"], alpha=0.7)
for k in range(n_classes):
    plt.scatter(X[y==k, 0], X[y==k, 1], color=colors[k], s=40, edgecolors="black", lw=0.5, label=f"class {k}")
plt.xlabel("$x_1$"); plt.ylabel("$x_2$"); plt.legend(); plt.title("Learned decision regions")
plt.show()


## ⭐ Stretch — ReLU vs sigmoid

Replace the hidden activation with **ReLU** — `relu(z) = max(0, z)`. The derivative is `(z > 0) * 1` (1 for positive z, 0 otherwise).

Train both networks side by side and plot loss curves. ReLU usually converges faster because the gradient doesn't saturate (sigmoid's gradient near 0 or 1 is tiny).

Note one wrinkle: for the ReLU derivative you need `z1` (the pre-activation), not `a1` (the post-activation). Adjust your backward function accordingly.


In [ ]:
# STRETCH: train the same network with ReLU instead of sigmoid as the hidden activation.
# Compare: which converges faster? Which reaches lower loss?

def relu(z):
    return np.maximum(0, z)

# TODO: copy your training loop, replace sigmoid with relu, replace the sigmoid derivative
# (a1 * (1 - a1)) with the relu derivative ((z1 > 0).astype(float)).
# Note: for relu, you need z1 (not a1) to compute the derivative. Plan around that.


## Wrap-up

You just implemented backprop from scratch. This is the most important algorithm in modern ML — it's literally what trains every neural network from MNIST classifiers to GPT-5.

Three things to remember:

1. **Backprop is the chain rule, run right-to-left.** Each layer needs the gradient flowing in from above; it then propagates a transformed gradient backward.
2. **Softmax + cross-entropy collapses to `(p - Y) / m`.** This is one of the cleanest results in ML — and the reason these two are paired everywhere.
3. **The forward outputs at each layer (a1, z1, etc.) are needed by backprop.** Don't discard them after forward; keep them around for the backward pass.

🎉 Course 2 Week 2 done. Next up: bias/variance, learning curves, and how to know which knob to turn when your model isn't working.
